# Évaluer un prompt de classification

Un prompt se met au point comme un modèle de *machine learning* : on le mesure sur un **jeu de test** dont on connaît les bonnes réponses, plutôt que de le juger sur deux ou trois essais.

Ici, un LLM trie les tickets du support informatique en cinq catégories. On compare plusieurs versions du prompt sur 40 tickets étiquetés à la main.

Le modèle : [Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B), un petit modèle ouvert sorti en 2026. Un GPU est recommandé (`Exécution > Modifier le type d'exécution`). Sans GPU, le notebook garde le même modèle, mais prend environ 30 minutes (durée indicative).

In [1]:
import logging
import warnings

import huggingface_hub
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore", message=r"(?s).*HF_TOKEN.*")
huggingface_hub.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

# Sans les paquets optionnels causal_conv1d et flash-linear-attention, transformers
# signale qu'il calcule les couches de Qwen3.5 en PyTorch pur : plus lent, même résultat.
logging.getLogger("transformers.integrations.hub_kernels").addFilter(
  lambda record: "falling back to its reference PyTorch" not in record.getMessage()
)

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cpu":
  warnings.warn(
    "Aucun GPU détecté : le notebook garde le même modèle, mais prend environ 30 minutes (durée indicative). "
    "Pour aller plus vite, choisissez un GPU (Exécution > Modifier le type d'exécution), puis réexécutez le notebook."
  )

LLM = "Qwen/Qwen3.5-2B"
DTYPE = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(LLM, padding_side="left")
modele = AutoModelForCausalLM.from_pretrained(LLM, dtype=DTYPE, device_map=device)

## Le jeu de test

40 tickets, 8 par catégorie, avec la catégorie qu'aurait choisie le support. Certains sont ambigus exprès : un e-mail d'hameçonnage parle de messagerie, mais relève de la sécurité ; un mot de passe VPN parle de réseau, mais relève de l'accès.

In [2]:
CATEGORIES = ["accès", "réseau", "messagerie", "poste", "sécurité"]

TICKETS = [
  ("Mon compte est verrouillé après plusieurs erreurs de mot de passe.", "accès"),
  ("Je n'ai pas les droits pour ouvrir le dossier partagé Finance.", "accès"),
  ("Mon mot de passe VPN a expiré, comment le changer ?", "accès"),
  ("Nouvel arrivant lundi : merci de créer son compte et sa boîte mail.", "accès"),
  (
    "Je ne reçois plus les codes de double authentification sur mon téléphone.",
    "accès",
  ),
  ("Peut-on me donner accès en écriture au wiki de l'équipe ?", "accès"),
  ("J'ai oublié mon mot de passe Windows.", "accès"),
  ("Outlook me redemande mon mot de passe à chaque démarrage.", "accès"),
  ("Le VPN se déconnecte toutes les dix minutes.", "réseau"),
  ("Pas de wifi dans la salle de réunion B.", "réseau"),
  ("Internet est très lent depuis ce matin sur tout l'étage.", "réseau"),
  ("Impossible de joindre le serveur de fichiers depuis l'agence de Lyon.", "réseau"),
  ("Le VPN refuse de se connecter : erreur « passerelle injoignable ».", "réseau"),
  ("Les visioconférences coupent sans arrêt depuis le site de Nantes.", "réseau"),
  ("La prise réseau de mon bureau ne fonctionne plus.", "réseau"),
  ("Le site intranet ne répond plus, pour tout le monde.", "réseau"),
  ("Je ne reçois plus d'e-mails depuis hier soir.", "messagerie"),
  ("Ma boîte mail est pleine, je ne peux plus envoyer de messages.", "messagerie"),
  ("Comment partager mon calendrier avec mon assistante ?", "messagerie"),
  ("Un client me dit que mes e-mails arrivent dans ses indésirables.", "messagerie"),
  ("Je voudrais une liste de diffusion pour l'équipe projet.", "messagerie"),
  ("Une pièce jointe de 30 Mo est refusée à l'envoi.", "messagerie"),
  ("Mes réponses automatiques d'absence ne partent pas.", "messagerie"),
  ("Les invitations aux réunions arrivent avec une heure de décalage.", "messagerie"),
  ("Mon écran reste noir au démarrage.", "poste"),
  ("L'imprimante du 3e étage imprime des pages blanches.", "poste"),
  ("J'aurais besoin d'Excel sur mon nouveau portable.", "poste"),
  ("Mon ordinateur est très lent depuis la dernière mise à jour.", "poste"),
  ("Plusieurs touches du clavier de mon portable ne répondent plus.", "poste"),
  ("La batterie de mon portable ne tient plus qu'une heure.", "poste"),
  ("Le logiciel de comptabilité plante à l'ouverture.", "poste"),
  ("Mon casque n'est pas reconnu par l'ordinateur.", "poste"),
  (
    "Un e-mail de la « DSI » me demande de confirmer mon mot de passe via un lien.",
    "sécurité",
  ),
  (
    "J'ai cliqué sur un lien dans un e-mail et une page m'a demandé mes identifiants.",
    "sécurité",
  ),
  ("L'antivirus signale un fichier infecté sur ma clé USB.", "sécurité"),
  ("J'ai perdu mon ordinateur portable professionnel dans le train.", "sécurité"),
  ("Des fenêtres publicitaires s'ouvrent toutes seules sur mon poste.", "sécurité"),
  ("Quelqu'un s'est connecté à mon compte depuis l'étranger cette nuit.", "sécurité"),
  (
    "Un fournisseur demande par e-mail de changer son RIB avant le prochain virement.",
    "sécurité",
  ),
  ("J'ai envoyé par erreur un fichier client à une adresse externe.", "sécurité"),
]
jeu = pd.DataFrame(TICKETS, columns=["ticket", "attendu"])
jeu.attendu.value_counts()

,count
attendu,
accès,8
réseau,8
messagerie,8
poste,8
sécurité,8


## Mesurer un prompt

La fonction `evaluer` envoie chaque ticket au modèle avec le prompt (`{ticket}` y est remplacé par le texte du ticket), lit la première catégorie citée dans la réponse, et calcule la proportion de bonnes réponses. Une réponse qui ne cite aucune catégorie compte comme une erreur, notée `?`.

In [3]:
def interroge(prompts: list[str], taille_lot: int = 8) -> list[str]:
  reponses = []
  for debut in range(0, len(prompts), taille_lot):
    messages = [
      [{"role": "user", "content": p}] for p in prompts[debut : debut + taille_lot]
    ]
    entrees = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      enable_thinking=False,
      padding=True,
      return_tensors="pt",
      return_dict=True,
    ).to(device)
    with torch.no_grad():
      sortie = modele.generate(
        **entrees,
        max_new_tokens=30,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
      )
    reponses += tokenizer.batch_decode(
      sortie[:, entrees["input_ids"].shape[1] :], skip_special_tokens=True
    )
  return reponses


def categorie(reponse: str) -> str:
  reponse = reponse.lower()
  trouvees = [(reponse.find(c), c) for c in CATEGORIES if c in reponse]
  return min(trouvees)[1] if trouvees else "?"


def evaluer(prompt: str) -> pd.DataFrame:
  reponses = interroge([prompt.replace("{ticket}", t) for t in jeu.ticket])
  resultats = jeu.assign(réponse=reponses, prédit=[categorie(a) for a in reponses])
  exactitude = (resultats.prédit == resultats.attendu).mean()
  print(
    f"Exactitude : {exactitude:.0%} ({(resultats.prédit == resultats.attendu).sum()}/{len(resultats)})"
  )
  return resultats


def erreurs(essai: pd.DataFrame) -> pd.DataFrame:
  return essai[essai.prédit != essai.attendu][["ticket", "attendu", "prédit"]]

## Version 1 : le prompt naïf

In [4]:
PROMPT_V1 = """Classe ce ticket dans une catégorie : accès, réseau, messagerie, poste ou sécurité.

Ticket : {ticket}"""

v1 = evaluer(PROMPT_V1)
v1.réponse.head()

Exactitude : 38% (15/40)


,réponse
0,Voici le ticket classé dans la catégorie **séc...
1,Voici le ticket classé dans la catégorie **séc...
2,Voici le ticket classé dans la catégorie **séc...
3,Voici le ticket classé dans la catégorie **Acc...
4,Voici le ticket classé dans la catégorie **séc...


Le modèle répond par des phrases de longueur variable : facile à lire pour un humain, moins pour un programme. Quelles erreurs fait-il ?

Dans l'exécution ci-dessous, 22 des 25 erreurs sont « sécurité », la dernière catégorie de la liste du prompt : le modèle penche vers une catégorie au lieu de peser chaque ticket, jusqu'à y ranger une imprimante ou une batterie. La colonne `réponse` montre que l'erreur vient bien du modèle, pas de la lecture de sa réponse : il écrit lui-même cette catégorie en gras.

In [5]:
from IPython.display import display

with pd.option_context("display.max_colwidth", None):
  display(erreurs(v1).join(v1.réponse))

,ticket,attendu,prédit,réponse
0,Mon compte est verrouillé après plusieurs erreurs de mot de passe.,accès,sécurité,Voici le ticket classé dans la catégorie **sécurité** :\n\n**Titre :** Mon compte est verrouillé après plusieurs erreurs de mot
1,Je n'ai pas les droits pour ouvrir le dossier partagé Finance.,accès,sécurité,Voici le ticket classé dans la catégorie **sécurité** :\n\n**Titre :** Accès refusé au dossier partagé Finance\n**Catégorie
2,"Mon mot de passe VPN a expiré, comment le changer ?",accès,sécurité,Voici le ticket classé dans la catégorie **sécurité** :\n\n**Catégorie :** Sécurité\n**Titre :** Mon mot de passe VPN
4,Je ne reçois plus les codes de double authentification sur mon téléphone.,accès,sécurité,Voici le ticket classé dans la catégorie **sécurité** :\n\n**Catégorie :** Sécurité\n**Titre :** Je ne reçois plus
5,Peut-on me donner accès en écriture au wiki de l'équipe ?,accès,sécurité,Voici le ticket classé dans la catégorie **sécurité** :\n\n**Titre :** Accès écriture au wiki de l'équipe\n**
6,J'ai oublié mon mot de passe Windows.,accès,sécurité,Voici le ticket classé dans la catégorie **sécurité** :\n\n**Catégorie :** Sécurité\n**Titre :** J'ai oublié mon mot
7,Outlook me redemande mon mot de passe à chaque démarrage.,accès,sécurité,Voici le ticket classé dans la catégorie **sécurité** :\n\n**Titre :** Outlook me redemande mon mot de passe à chaque
8,Le VPN se déconnecte toutes les dix minutes.,réseau,sécurité,Voici le ticket classé dans la catégorie **sécurité** :\n\n**Catégorie :** Sécurité\n**Titre :** Le VPN se déconnect
9,Pas de wifi dans la salle de réunion B.,réseau,sécurité,Voici le ticket classé dans la catégorie **sécurité** (car il concerne la protection de l'infrastructure réseau et la disponibilité des équipements
10,Internet est très lent depuis ce matin sur tout l'étage.,réseau,accès,Voici le ticket classé dans la catégorie **Accès** :\n\n**Catégorie :** Accès\n**Description :** Internet est très lent depuis ce


## Version 2 : définir les catégories et le format

Le modèle ne connaît pas nos conventions : où ranger un mot de passe VPN, ou un e-mail d'hameçonnage ? On définit chaque catégorie, avec les règles de priorité, et on impose le format de la réponse.

In [6]:
PROMPT_V2 = """Tu tries les tickets du support informatique. Catégories :
- accès : comptes, mots de passe (y compris celui du VPN ou de la messagerie), droits, double authentification
- réseau : VPN, wifi, connexion internet, lenteur ou coupure du réseau, sites internes injoignables
- messagerie : e-mails, calendrier, listes de diffusion, pièces jointes
- poste : matériel et logiciels du poste de travail, imprimantes
- sécurité : e-mail suspect ou frauduleux, virus, vol ou perte de matériel, connexion suspecte, fuite de données

Règle : tout ce qui touche à une menace ou à un incident de sécurité relève de « sécurité », même si cela passe par un e-mail ou un compte.

Réponds uniquement par le nom de la catégorie, en minuscules.

Ticket : {ticket}"""

v2 = evaluer(PROMPT_V2)
erreurs(v2)

Exactitude : 78% (31/40)


,ticket,attendu,prédit
21,Une pièce jointe de 30 Mo est refusée à l'envoi.,messagerie,sécurité
22,Mes réponses automatiques d'absence ne partent...,messagerie,sécurité
26,J'aurais besoin d'Excel sur mon nouveau portable.,poste,?
27,Mon ordinateur est très lent depuis la dernièr...,poste,?
29,La batterie de mon portable ne tient plus qu'u...,poste,?
31,Mon casque n'est pas reconnu par l'ordinateur.,poste,?
32,Un e-mail de la « DSI » me demande de confirme...,sécurité,accès
33,J'ai cliqué sur un lien dans un e-mail et une ...,sécurité,accès
35,J'ai perdu mon ordinateur portable professionn...,sécurité,?


## Version 3 : ajouter des exemples

On ajoute quelques exemples résolus, choisis parmi les cas difficiles, mais **absents du jeu de test** : sinon, on mesurerait la mémoire du modèle, pas sa capacité à généraliser.

In [7]:
EXEMPLES = """Exemples :
Ticket : Mon accès au VPN est bloqué après trois mauvais mots de passe. → accès
Ticket : Le partage d'écran se fige pendant les réunions à distance avec l'agence. → réseau
Ticket : Je ne retrouve plus mes anciens e-mails archivés. → messagerie
Ticket : Ma souris sans fil ne fonctionne plus. → poste
Ticket : Un SMS au nom de la banque de l'entreprise me demande mes identifiants. → sécurité

"""

PROMPT_V3 = PROMPT_V2.replace("Ticket : {ticket}", EXEMPLES + "Ticket : {ticket}")

v3 = evaluer(PROMPT_V3)
erreurs(v3)

Exactitude : 85% (34/40)


,ticket,attendu,prédit
4,Je ne reçois plus les codes de double authenti...,accès,sécurité
19,Un client me dit que mes e-mails arrivent dans...,messagerie,sécurité
21,Une pièce jointe de 30 Mo est refusée à l'envoi.,messagerie,sécurité
23,Les invitations aux réunions arrivent avec une...,messagerie,réseau
27,Mon ordinateur est très lent depuis la dernièr...,poste,réseau
36,Des fenêtres publicitaires s'ouvrent toutes se...,sécurité,poste


## Comparer les versions

Les bonnes réponses par catégorie, pour chaque version :

In [8]:
pd.DataFrame(
  {
    version: (essai.prédit == essai.attendu).groupby(essai.attendu).sum()
    for version, essai in {"v1": v1, "v2": v2, "v3": v3}.items()
  }
)

,v1,v2,v3
attendu,,,
accès,1,8,7
messagerie,5,6,5
poste,0,4,7
réseau,1,8,8
sécurité,8,5,7


Chaque version corrige des erreurs et peut en créer d'autres. Dans l'exécution ci-dessus, avec ses définitions, la v2 fait passer l'accès et le réseau de 1 à 8 bonnes réponses sur 8, mais la sécurité tombe de 8 à 5 : la v1 la réussissait parce qu'elle répondait « sécurité » à presque tout, et la v2 range deux hameçonnages dans « accès ». Les exemples de la v3 redressent le poste, de 4 à 7, et la sécurité, de 5 à 7, mais un problème de compte (les codes de double authentification) part à son tour dans « sécurité », et la messagerie retombe à 5, son niveau de la v1. De la v2 à la v3, le total ne gagne que 3 tickets, et cache des pertes : d'où l'intérêt de mesurer catégorie par catégorie, pas seulement au total.

La matrice de confusion de la dernière version (v3) : en ligne la catégorie attendue, en colonne la catégorie prédite.

In [9]:
pd.crosstab(v3.attendu, v3.prédit)

prédit,accès,messagerie,poste,réseau,sécurité
attendu,,,,,
accès,7,0,0,0,1
messagerie,0,5,0,1,2
poste,0,0,7,1,0
réseau,0,0,0,8,0
sécurité,0,0,1,0,7


Avec 40 tickets, un ticket vaut 2,5 points : un écart de 5 points entre deux versions peut tenir au hasard. Un vrai jeu de test compte plutôt une centaine de cas, tirés des tickets réels.

## À vous

Écrivez votre propre version du prompt et mesurez-la. La cellule part de la v3 : modifiez `MON_PROMPT` pour redresser sa catégorie la plus faible, la messagerie (5 sur 8 ci-dessus : une pièce jointe refusée et des e-mails qui arrivent en indésirables partent dans « sécurité »), par exemple avec une règle ou un exemple de plus, en gardant `{ticket}` à l'endroit du ticket. Sans GPU, comptez environ 9 minutes par essai (durée indicative).

In [10]:
# À modifier, par exemple :
# MON_PROMPT = PROMPT_V3.replace("Réponds uniquement", "Règle : …\n\nRéponds uniquement")
MON_PROMPT = PROMPT_V3

mien = evaluer(MON_PROMPT)
erreurs(mien)

Exactitude : 85% (34/40)


,ticket,attendu,prédit
4,Je ne reçois plus les codes de double authenti...,accès,sécurité
19,Un client me dit que mes e-mails arrivent dans...,messagerie,sécurité
21,Une pièce jointe de 30 Mo est refusée à l'envoi.,messagerie,sécurité
23,Les invitations aux réunions arrivent avec une...,messagerie,réseau
27,Mon ordinateur est très lent depuis la dernièr...,poste,réseau
36,Des fenêtres publicitaires s'ouvrent toutes se...,sécurité,poste


## À retenir

- Un prompt se juge sur un jeu de test étiqueté, pas sur quelques essais
- Définir les catégories et nos conventions, imposer le format, donner des exemples : chaque ajout se mesure
- Les exemples du prompt ne doivent pas venir du jeu de test
- Ce jeu de test se rejoue à chaque changement de prompt ou de modèle